# Model 2: Random Forest for LOS/NLOS classification

This notebook reproduces the **Random Forest (RF)** model from the paper (Section 3.3.3) on the full-resolution preprocessed data in `../output_raw_full/`. It is built the same way as `../SVM/svm_los_nlos.ipynb`, so the two models can be compared directly.

| | Paper | This notebook |
|---|---|---|
| Classifier | Random Forest, 100 decision trees, Gini impurity | same |
| Input | 72 values per frame (mean \|I/Q\| over raw samples) | **3328 values per frame** (CSI amplitude, 64 antennas × 52 subcarriers) |
| Split | cluster-wise: 2 LOS + 3 NLOS train, 2 LOS + 2 NLOS test | same clusters (Part A) + **all 80 splits** (Part B) |
| Normalisation | z-score per file (test files included) | per-antenna normalisation (`NORM = "antenna"`) + z-score fitted on training clusters only |
| Metrics | accuracy, confusion matrix | accuracy, confusion matrix, precision, recall, F1, ROC-AUC, PR curve, probability distribution, accuracy per cluster, feature importance |

The notebook has two parts:
- **Part A (Steps 1–14): the paper's single split.** Train on LOS 1, LOS 2, NLOS 1–3; test on LOS 3, LOS 4, NLOS 4, NLOS 5.
- **Part B (Steps 15–21): all splits.** 20 leave-one-pair-out splits + 60 paper-style splits, and a comparison with the SVM.

**Steps (Part A)**
1. Setup
2. Load the data and check the preprocessing settings
3. Train the Random Forest
4. Predict on the unseen test clusters
5. Classification metrics
6. Accuracy per test cluster
7. Confusion matrix
8. ROC curve
9. Precision–recall curve
10. Probability distribution
11. Feature importance (which antennas and subcarriers the forest uses)
12. Overfitting check: training vs out-of-bag vs test accuracy
13. Comparison with the paper (RF) and with the SVM
14. Save the model and results

**Steps (Part B)**

15. List all splits
16. Load the per-cluster features
17. Evaluation function
18. Run all splits (progress saved after every split)
19. Summary: mean ± std, worst split, accuracy per held-out cluster
20. Plots and saved summary
21. Comparison with the SVM on the same splits

## Step 1: Setup

The paper states 100 trees and the Gini criterion. The other settings are scikit-learn defaults: `max_features = "sqrt"` (about 58 of the 3328 features are tried at each split), and trees are grown until their leaves are pure.

**Running in Colab:** set `DATA_DIR` to the `output_raw_full` folder on your Drive.

In [ ]:
import os
import json
import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, average_precision_score, classification_report,
                             confusion_matrix, precision_recall_curve, roc_auc_score, roc_curve)

DATA_DIR = "../output_raw_full"
RESULTS_DIR = "results"
SVM_RESULTS_DIR = "../SVM/results"   # for the comparison steps (optional)

N_ESTIMATORS = 100      # paper: 100 decision trees
CRITERION = "gini"      # paper: Gini impurity
MAX_FEATURES = "sqrt"   # scikit-learn default
SEED = 42
N_JOBS = -1             # use all CPU cores

CLASS_NAMES = ["NLOS (0)", "LOS (1)"]

# chart colours (reference palette): one colour per class, text in neutral ink
COLOR_LOS = "#2a78d6"
COLOR_NLOS = "#eb6834"
INK = "#0b0b0b"
INK_2 = "#52514e"
GRID = "#e4e3df"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": INK_2, "axes.labelcolor": INK, "text.color": INK,
    "xtick.color": INK_2, "ytick.color": INK_2,
    "axes.grid": True, "grid.color": GRID, "grid.linewidth": 0.8,
    "axes.spines.top": False, "axes.spines.right": False,
    "font.size": 11, "lines.linewidth": 2,
})

os.makedirs(RESULTS_DIR, exist_ok=True)

## Step 2: Load the data and check the preprocessing settings

`config.json` (written by `preprocess_raw.ipynb`) must show `norm = "antenna"` and `feature = "full"`. Otherwise, re-run the preprocessing first.

| Set | Clusters |
|---|---|
| Train | LOS 1, LOS 2, NLOS 1, NLOS 2, NLOS 3 |
| Test (unseen) | LOS 3, LOS 4, NLOS 4, NLOS 5 |

In [ ]:
with open(os.path.join(DATA_DIR, "config.json")) as f:
    prep = json.load(f)
print("preprocessing:", {k: prep.get(k) for k in ["feature", "norm", "n_features", "window", "stride", "skip_first_frames"]})
assert prep.get("norm") == "antenna", "data is not per-antenna normalised -- run preprocess_raw.ipynb with NORM = 'antenna'"
assert prep.get("feature") == "full", "expected FEATURE = 'full' (3328 values per frame)"
print()

X_train = np.load(os.path.join(DATA_DIR, "X_train_frames.npy"))
y_train = np.load(os.path.join(DATA_DIR, "y_train_frames.npy"))
g_train = np.load(os.path.join(DATA_DIR, "groups_train_frames.npy"))

X_test = np.load(os.path.join(DATA_DIR, "X_test_frames.npy"))
y_test = np.load(os.path.join(DATA_DIR, "y_test_frames.npy"))
g_test = np.load(os.path.join(DATA_DIR, "groups_test_frames.npy"))

for name, X, y, g in [("train", X_train, y_train, g_train), ("test", X_test, y_test, g_test)]:
    print(f"{name}: X={X.shape} {X.dtype}  LOS={int((y == 1).sum())}  NLOS={int((y == 0).sum())}")
    for c, n in zip(*np.unique(g, return_counts=True)):
        print(f"    {c:14s} {n:5d} frames")

## Step 3: Train the Random Forest

A Random Forest trains many decision trees, each on a random bootstrap sample of the training frames and with a random subset of features at each split. The prediction is the majority vote of the trees (Eq. 8 in the paper uses the Gini impurity to choose splits):

$$G = 1 - \sum_{i=1}^{n} p_i^2$$

`oob_score=True` also computes the **out-of-bag (OOB) accuracy**: each frame is scored only by the trees that did not see it in training. It is used in Step 12 to check for overfitting.

In [ ]:
rf = RandomForestClassifier(n_estimators=N_ESTIMATORS, criterion=CRITERION, max_features=MAX_FEATURES,
                            oob_score=True, n_jobs=N_JOBS, random_state=SEED)

t0 = time.time()
rf.fit(X_train, y_train)
train_time = time.time() - t0

depths = [t.get_depth() for t in rf.estimators_]
leaves = [t.get_n_leaves() for t in rf.estimators_]
print(f"training time: {train_time:.1f} s")
print(f"tree depth: mean {np.mean(depths):.1f} (max {max(depths)})   leaves per tree: mean {np.mean(leaves):.0f}")
print(f"out-of-bag accuracy: {rf.oob_score_:.4f}")

## Step 4: Predict on the unseen test clusters

`predict_proba` gives P(LOS) as the fraction of trees voting LOS; `predict` uses the 0.5 threshold.

In [ ]:
t0 = time.time()
p_los = rf.predict_proba(X_test)[:, 1]
y_pred = (p_los >= 0.5).astype(np.int8)
pred_time = time.time() - t0

print(f"prediction time for {len(y_test)} frames: {pred_time:.1f} s")

## Step 5: Classification metrics

In [ ]:
acc = accuracy_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, p_los)
pr_auc = average_precision_score(y_test, p_los)

print(f"Accuracy : {acc:.4f}")
print(f"ROC-AUC  : {roc_auc:.4f}")
print(f"PR-AUC   : {pr_auc:.4f}  (average precision, LOS as positive class)")
print()
print(classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4))
report = classification_report(y_test, y_pred, target_names=CLASS_NAMES, digits=4, output_dict=True)

## Step 6: Accuracy per test cluster

Each test cluster is a separate unseen environment. A single overall accuracy can hide one cluster that fails, so each cluster is checked separately.

In [ ]:
per_cluster = {}
print(f"{'cluster':14s} {'label':>5s} {'frames':>6s} {'accuracy':>9s} {'mean P(LOS)':>12s}")
for c in np.unique(g_test):
    m = g_test == c
    per_cluster[c] = {
        "label": int(y_test[m][0]),
        "frames": int(m.sum()),
        "accuracy": float(accuracy_score(y_test[m], y_pred[m])),
        "mean_p_los": float(p_los[m].mean()),
    }
    r = per_cluster[c]
    print(f"{c:14s} {r['label']:5d} {r['frames']:6d} {r['accuracy']:9.4f} {r['mean_p_los']:12.4f}")

## Step 7: Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm_pct = cm / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(5.2, 4.4))
ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)
ax.grid(False)
for i in range(2):
    for j in range(2):
        ax.text(j, i, f"{cm[i, j]:,}\n({cm_pct[i, j]:.1f}%)", ha="center", va="center",
                color="white" if cm_pct[i, j] > 60 else INK, fontsize=12)
ax.set_xticks([0, 1], CLASS_NAMES)
ax.set_yticks([0, 1], CLASS_NAMES)
ax.set_xlabel("Predicted label")
ax.set_ylabel("True label")
ax.set_title("RF: confusion matrix (unseen test clusters)")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "confusion_matrix.png"), dpi=150)
plt.show()

print(cm)

## Step 8: ROC curve

In [ ]:
fpr, tpr, _ = roc_curve(y_test, p_los)

fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.plot(fpr, tpr, color=COLOR_LOS, label=f"RF (AUC = {roc_auc:.4f})")
ax.plot([0, 1], [0, 1], color=INK_2, linestyle="--", linewidth=1, label="Random classifier")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.01)
ax.set_xlabel("False positive rate")
ax.set_ylabel("True positive rate")
ax.set_title("RF: ROC curve (LOS = positive class)")
ax.legend(loc="lower right", frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "roc_curve.png"), dpi=150)
plt.show()

## Step 9: Precision–recall curve

In [ ]:
prec, rec, _ = precision_recall_curve(y_test, p_los)
baseline = y_test.mean()

fig, ax = plt.subplots(figsize=(5.2, 4.6))
ax.plot(rec, prec, color=COLOR_LOS, label=f"RF (AP = {pr_auc:.4f})")
ax.axhline(baseline, color=INK_2, linestyle="--", linewidth=1, label=f"Chance (LOS share = {baseline:.2f})")
ax.set_xlim(0, 1)
ax.set_ylim(0, 1.01)
ax.set_xlabel("Recall")
ax.set_ylabel("Precision")
ax.set_title("RF: precision–recall curve (LOS = positive class)")
ax.legend(loc="lower left", frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "pr_curve.png"), dpi=150)
plt.show()

## Step 10: Probability distribution

The histogram shows the predicted P(LOS), the share of trees voting LOS, for frames that are truly NLOS and truly LOS. The dashed line is the 0.5 threshold.

In [ ]:
bins = np.linspace(0, 1, 41)
fig, ax = plt.subplots(figsize=(6.4, 4.2))
ax.hist(p_los[y_test == 0], bins=bins, color=COLOR_NLOS, alpha=0.75, label="True NLOS (0)",
        edgecolor="#fcfcfb", linewidth=1)
ax.hist(p_los[y_test == 1], bins=bins, color=COLOR_LOS, alpha=0.75, label="True LOS (1)",
        edgecolor="#fcfcfb", linewidth=1)
ax.axvline(0.5, color=INK_2, linestyle="--", linewidth=1)
ax.set_yscale("log")
ax.set_xlabel("Predicted P(LOS)  (share of trees voting LOS)")
ax.set_ylabel("Number of frames (log scale)")
ax.set_title("RF: predicted probability by true class")
ax.legend(frameon=False)
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "probability_distribution.png"), dpi=150)
plt.show()

## Step 11: Feature importance

The forest records how much each of the 3328 features reduces the Gini impurity. Reshaping them to (64 antennas × 52 subcarriers) shows **where in the channel** the forest finds LOS/NLOS information:
- the **heatmap** shows each antenna × subcarrier value;
- the **subcarrier profile** sums over antennas, showing which parts of the band matter;
- the **antenna profile** sums over subcarriers. After per-antenna normalisation it should be fairly flat. A few dominant antennas would suggest location-specific cues.

**Watch subcarrier 13 (802.11 k = −13).** The raw data has a narrowband **interference spur** there, also leaking into subcarriers 12 and 14. It shows up even in the noise-only slot: +16.5 dB above the other subcarriers in every NLOS file, but only +4.9 dB in every LOS file. Because NLOS was recorded with 30 dB more gain and lower SNR, the spur makes a visible bump in every NLOS frame. That bump separates the classes perfectly but isn't physical. If the importance concentrates at subcarrier 13, the forest is using this artefact.

In [ ]:
imp = rf.feature_importances_.reshape(64, 52)
imp_sub = imp.sum(axis=0)     # per subcarrier
imp_ant = imp.sum(axis=1)     # per antenna

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2), gridspec_kw={"width_ratios": [1.3, 1, 1]})
ax = axes[0]
im = ax.imshow(imp, aspect="auto", cmap="Blues", interpolation="nearest")
ax.grid(False)
ax.set_xlabel("Subcarrier index (0–51)")
ax.set_ylabel("Antenna index (0–63)")
ax.set_title("Importance per antenna × subcarrier")
fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

axes[1].plot(np.arange(52), imp_sub, color=COLOR_LOS)
axes[1].set_xlabel("Subcarrier index")
axes[1].set_ylabel("Importance (sum over antennas)")
axes[1].set_title("Importance per subcarrier")

axes[2].plot(np.arange(64), imp_ant, color=COLOR_LOS)
axes[2].axhline(1 / 64, color=INK_2, linestyle="--", linewidth=1)
axes[2].text(63, 1 / 64, "uniform", ha="right", va="bottom", color=INK_2)
axes[2].set_xlabel("Antenna index")
axes[2].set_ylabel("Importance (sum over subcarriers)")
axes[2].set_title("Importance per antenna")
fig.tight_layout()
fig.savefig(os.path.join(RESULTS_DIR, "feature_importance.png"), dpi=150)
plt.show()

top = np.argsort(rf.feature_importances_)[::-1][:10]
print("top 10 features (antenna, subcarrier, importance):")
for k in top:
    print(f"  antenna {k // 52:2d}, subcarrier {k % 52:2d}: {rf.feature_importances_[k]:.4f}")
print(f"\nshare of importance in the 10 most important antennas: {np.sort(imp_ant)[::-1][:10].sum():.3f} (uniform would be {10/64:.3f})")

## Step 12: Overfitting check

| Accuracy | Measured on | What it tells you |
|---|---|---|
| Training | the training frames | fully grown trees usually reach about 100%; on its own this says nothing |
| Out-of-bag (OOB) | training frames, each scored only by trees that didn't see it | generalisation to **new frames from the same environments** |
| Test | frames from **unseen environments** | generalisation to **new environments**, the paper's question |

A large gap between OOB and test means the forest generalises within the training environments but not to new ones.

In [ ]:
train_acc = accuracy_score(y_train, rf.predict(X_train))
print(f"training accuracy   : {train_acc:.4f}")
print(f"out-of-bag accuracy : {rf.oob_score_:.4f}   (new frames, same environments)")
print(f"test accuracy       : {acc:.4f}   (unseen environments)")
print(f"gap OOB - test      : {rf.oob_score_ - acc:+.4f}")

## Step 13: Comparison with the paper (RF) and with the SVM

| Column | Data |
|---|---|
| `paper` | the paper's RF (Section 4.2): accuracy 53.82%, AUC 0.94. Precision, recall and F1 are **derived from the paper's confusion matrix** (8,099 / 8,120 NLOS and 4,998 / 16,213 LOS correct), because the text gives them only partly. |
| `SVM` | this project's SVM on the same data and split (`../SVM/results/metrics.json`), if available |
| `RF (this run)` | this notebook |

The paper's inputs are different (72 averaged raw-sample values), so its column is for reference only.

In [ ]:
paper = {
    "accuracy": 0.5382, "roc_auc": 0.94,
    "NLOS (0)": {"precision": 8099 / (8099 + 11215), "recall": 8099 / 8120, "support": 8120},
    "LOS (1)": {"precision": 4998 / (4998 + 21), "recall": 4998 / 16213, "support": 16213},
}
for cls in CLASS_NAMES:
    p_, r_ = paper[cls]["precision"], paper[cls]["recall"]
    paper[cls]["f1-score"] = 2 * p_ * r_ / (p_ + r_)

svm_path = os.path.join(SVM_RESULTS_DIR, "metrics.json")
svm = json.load(open(svm_path)) if os.path.exists(svm_path) else None
if svm is None:
    print("(SVM results not found -- run ../SVM/svm_los_nlos.ipynb Part A to include them)\n")

def svm_val(key, cls=None):
    if svm is None:
        return float("nan")
    return svm[key] if cls is None else svm["classification_report"][cls][key]

rows = [("Accuracy", paper["accuracy"], svm_val("accuracy"), acc),
        ("ROC-AUC", paper["roc_auc"], svm_val("roc_auc"), roc_auc)]
for cls in CLASS_NAMES:
    for k in ["precision", "recall", "f1-score"]:
        rows.append((f"{cls} {k}", paper[cls][k], svm_val(k, cls), report[cls][k]))
for cls in CLASS_NAMES:
    rows.append((f"{cls} support", paper[cls]["support"], svm_val("support", cls), report[cls]["support"]))

print(f"{'metric':22s} {'paper RF':>10s} {'SVM':>10s} {'RF (this run)':>14s}")
for name, p_, s_, o_ in rows:
    fmt = "{:10.0f}" if "support" in name else "{:10.4f}"
    print(f"{name:22s} {fmt.format(p_)} {fmt.format(s_)} {fmt.format(o_):>14s}")

if svm is not None:
    print()
    print(f"{'accuracy per cluster':22s} {'SVM':>10s} {'RF (this run)':>14s}")
    for c in per_cluster:
        print(f"{c:22s} {svm['per_cluster'][c]['accuracy']:10.4f} {per_cluster[c]['accuracy']:14.4f}")

## Step 14: Save the model and results

| File | Contents |
|---|---|
| `results/rf_model.joblib` | the trained forest (compressed; load with `joblib.load`) |
| `results/metrics.json` | all metrics, settings and timings |
| `results/predictions.npz` | `y_true`, `y_pred`, `p_los` and the cluster name for every test frame |
| `results/feature_importance.npy` | importance per feature, shape (64, 52) |
| `results/*.png` | the figures |

In [ ]:
joblib.dump(rf, os.path.join(RESULTS_DIR, "rf_model.joblib"), compress=3)
np.save(os.path.join(RESULTS_DIR, "feature_importance.npy"), imp)
np.savez(os.path.join(RESULTS_DIR, "predictions.npz"),
         y_true=y_test, y_pred=y_pred, p_los=p_los, groups=g_test)

metrics = {
    "model": "RandomForestClassifier",
    "params": {"n_estimators": N_ESTIMATORS, "criterion": CRITERION, "max_features": MAX_FEATURES,
               "random_state": SEED},
    "data": {"dir": DATA_DIR, "n_features": int(X_train.shape[1]), "norm": prep.get("norm"),
             "n_train": int(len(y_train)), "n_test": int(len(y_test))},
    "accuracy": acc, "roc_auc": roc_auc, "pr_auc": pr_auc,
    "train_accuracy": train_acc, "oob_accuracy": rf.oob_score_,
    "classification_report": report,
    "confusion_matrix": cm.tolist(),
    "per_cluster": per_cluster,
    "tree_depth_mean": float(np.mean(depths)), "leaves_mean": float(np.mean(leaves)),
    "train_time_s": train_time, "predict_time_s": pred_time,
}
with open(os.path.join(RESULTS_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)

for fn in sorted(os.listdir(RESULTS_DIR)):
    print(f"{fn:28s} {os.path.getsize(os.path.join(RESULTS_DIR, fn)) / 1e6:7.2f} MB")

---
# Part B: evaluation on all splits

## Step 15: List all splits

With 4 LOS and 5 NLOS clusters, two protocols are used (the same as for the SVM):

| Protocol | Test set per split | Train set per split | Number of splits |
|---|---|---|---|
| `lopo`: leave-one-pair-out | 1 LOS + 1 NLOS cluster | the other 7 clusters | 4 × 5 = **20** |
| `paper_style`: all paper-like splits | 2 LOS + 2 NLOS clusters | the other 2 LOS + 3 NLOS | C(4,2) × C(5,2) = 6 × 10 = **60** |

The paper's split (test = LOS 3, LOS 4, NLOS 4, NLOS 5) is one of the 60 `paper_style` splits.

**Settings:**
- `PROTOCOLS`: which protocols to run.
- `TRAIN_SUBSAMPLE`: `None` uses **all** training frames of each split. Set a number, for example `10000`, for a faster run.

**Runtime with all frames** (estimate for this Mac, 10 cores): about 30–45 s per split, so about **15 min for `lopo`** and **about 30 min for `paper_style`**. Progress is saved after each split. If the run stops, run Step 18 again and it continues where it left off.

**To run only Part B:** run Step 1, then Steps 15–21.

In [ ]:
import itertools

LOS_CLUSTERS = ["LOS_cluster1", "LOS_cluster2", "LOS_cluster3", "LOS_cluster4"]
NLOS_CLUSTERS = ["NLOS_cluster1", "NLOS_cluster2", "NLOS_cluster3", "NLOS_cluster4", "NLOS_cluster5"]
ALL_CLUSTERS = LOS_CLUSTERS + NLOS_CLUSTERS
PAPER_TEST = ["LOS_cluster3", "LOS_cluster4", "NLOS_cluster4", "NLOS_cluster5"]

PROTOCOLS = ["lopo", "paper_style"]   # run both; remove one to save time
TRAIN_SUBSAMPLE = None                # None = all training frames; e.g. 10000 for a fast run

short = lambda c: c.replace("_cluster", "")

def make_splits(protocol):
    if protocol == "lopo":
        tests = [[l, n] for l in LOS_CLUSTERS for n in NLOS_CLUSTERS]
    elif protocol == "paper_style":
        tests = [list(ls) + list(ns) for ls in itertools.combinations(LOS_CLUSTERS, 2)
                 for ns in itertools.combinations(NLOS_CLUSTERS, 2)]
    else:
        raise ValueError(protocol)
    splits = []
    for test in tests:
        train = [c for c in ALL_CLUSTERS if c not in test]
        splits.append({"name": "test=" + "+".join(short(c) for c in test), "train": train, "test": test})
    return splits

SPLITS = {p: make_splits(p) for p in PROTOCOLS}
for p, splits in SPLITS.items():
    print(f"{p}: {len(splits)} splits")
    for i, sp in enumerate(splits, 1):
        mark = "   <- paper split" if sorted(sp["test"]) == sorted(PAPER_TEST) else ""
        print(f"  {i:2d}. train={'+'.join(short(c) for c in sp['train']):38s} {sp['name']}{mark}")

## Step 16: Load the per-cluster features

The per-cluster features in `../output_raw_full/clusters/` are loaded (per-antenna normalised, saved **before** the z-score; about 0.7 GB in memory). The ready-made `X_*_frames.npy` files can't be used here, because their z-score was fitted on the paper's training clusters.

In [ ]:
CLUSTER_DIR = os.path.join(DATA_DIR, "clusters")
with open(os.path.join(CLUSTER_DIR, "index.json")) as f:
    cidx = json.load(f)
assert cidx["norm"] == "antenna" and not cidx["zscore_applied"], "re-run preprocess_raw.ipynb with NORM = 'antenna'"

cluster_X = {c: np.load(os.path.join(CLUSTER_DIR, cidx["clusters"][c]["features_file"])) for c in ALL_CLUSTERS}
cluster_y = {c: cidx["clusters"][c]["label"] for c in ALL_CLUSTERS}
for c in ALL_CLUSTERS:
    print(f"{c:14s} label={cluster_y[c]}  X={cluster_X[c].shape}")

## Step 17: Evaluation function

For one split:
1. stack the training clusters, optionally subsample them (`TRAIN_SUBSAMPLE`);
2. train the forest with the same settings as Part A;
3. predict the test frames and compute accuracy, macro-F1, ROC-AUC, recall per class and accuracy per test cluster.

**No z-score is needed here.** A decision tree only compares each feature with a threshold. Shifting and scaling a feature moves the threshold but doesn't change which frames go left or right, so the forest's predictions are the same with or without the z-score. Skipping it saves time and memory. (The SVM needs it; the forest doesn't.)

In [ ]:
from sklearn.metrics import f1_score, recall_score

def evaluate_split(split, seed=SEED):
    rng = np.random.default_rng(seed)
    Xtr = np.concatenate([cluster_X[c] for c in split["train"]])
    ytr = np.concatenate([np.full(len(cluster_X[c]), cluster_y[c], dtype=np.int8) for c in split["train"]])
    if TRAIN_SUBSAMPLE and TRAIN_SUBSAMPLE < len(ytr):
        keep = np.sort(rng.choice(len(ytr), TRAIN_SUBSAMPLE, replace=False))
        Xtr, ytr = Xtr[keep], ytr[keep]

    model = RandomForestClassifier(n_estimators=N_ESTIMATORS, criterion=CRITERION, max_features=MAX_FEATURES,
                                   n_jobs=N_JOBS, random_state=seed)
    t0 = time.time()
    model.fit(Xtr, ytr)
    fit_s = time.time() - t0
    del Xtr

    y_true, y_hat, score, per_cluster_acc = [], [], [], {}
    for c in split["test"]:
        p = model.predict_proba(cluster_X[c])[:, 1]
        pred = (p >= 0.5).astype(np.int8)
        yt = np.full(len(pred), cluster_y[c], dtype=np.int8)
        per_cluster_acc[c] = float((pred == yt).mean())
        y_true.append(yt); y_hat.append(pred); score.append(p)
    y_true, y_hat, score = np.concatenate(y_true), np.concatenate(y_hat), np.concatenate(score)

    return {
        "name": split["name"], "train": split["train"], "test": split["test"],
        "n_train": int(len(ytr)), "n_test": int(len(y_true)),
        "accuracy": float((y_hat == y_true).mean()),
        "macro_f1": float(f1_score(y_true, y_hat, average="macro")),
        "roc_auc": float(roc_auc_score(y_true, score)),
        "recall_los": float(recall_score(y_true, y_hat, pos_label=1)),
        "recall_nlos": float(recall_score(y_true, y_hat, pos_label=0)),
        "per_cluster_accuracy": per_cluster_acc,
        "fit_seconds": fit_s,
    }

## Step 18: Run all splits

Results are appended to `results/splits_<protocol>.jsonl`, one line per split. Splits that are already in the file are skipped, so the run can be resumed after a stop. To start over (for example after changing a setting), delete those files.

In [ ]:
def load_done(path):
    if not os.path.exists(path):
        return {}
    with open(path) as f:
        return {r["name"]: r for r in (json.loads(line) for line in f if line.strip())}

ALL_RESULTS = {}
for protocol, splits in SPLITS.items():
    path = os.path.join(RESULTS_DIR, f"splits_{protocol}.jsonl")
    done = load_done(path)
    print(f"=== {protocol}: {len(splits)} splits ({len(done)} already done)")
    t_start = time.time()
    for i, sp in enumerate(splits, 1):
        if sp["name"] in done:
            continue
        r = evaluate_split(sp)
        with open(path, "a") as f:
            f.write(json.dumps(r) + "\n")
        done[r["name"]] = r
        print(f"  [{i:2d}/{len(splits)}] {r['name']:32s} acc={r['accuracy']:.4f}  macro-F1={r['macro_f1']:.4f}  "
              f"AUC={r['roc_auc']:.4f}  ({r['fit_seconds']:.0f}s fit, {time.time() - t_start:.0f}s total)")
    ALL_RESULTS[protocol] = [done[sp["name"]] for sp in splits]

## Step 19: Summary

- **Mean ± std over splits** is the main number to report. The worst split shows how badly the model can do on an unlucky set of environments.
- **Accuracy of each held-out cluster**, averaged over every split where it was in the test set, shows which environments are hard.
- For `paper_style`, the paper's split is ranked among all 60 to show whether it is typical.

In [ ]:
SUMMARY = {}
for protocol, res in ALL_RESULTS.items():
    acc_s = np.array([r["accuracy"] for r in res])
    f1_s = np.array([r["macro_f1"] for r in res])
    auc_s = np.array([r["roc_auc"] for r in res])
    worst = res[int(acc_s.argmin())]
    print(f"=== {protocol} ({len(res)} splits)")
    print(f"  accuracy : {acc_s.mean():.4f} ± {acc_s.std():.4f}   min {acc_s.min():.4f}   max {acc_s.max():.4f}")
    print(f"  macro-F1 : {f1_s.mean():.4f} ± {f1_s.std():.4f}")
    print(f"  ROC-AUC  : {auc_s.mean():.4f} ± {auc_s.std():.4f}")
    print(f"  worst split: {worst['name']}  (acc {worst['accuracy']:.4f})")

    held = {c: [r["per_cluster_accuracy"][c] for r in res if c in r["per_cluster_accuracy"]] for c in ALL_CLUSTERS}
    held = {c: v for c, v in held.items() if v}
    print("  accuracy of each cluster when held out (mean / worst / number of splits):")
    for c in held:
        print(f"    {c:14s} {np.mean(held[c]):.4f} / {np.min(held[c]):.4f} / {len(held[c])}")

    entry = {"n_splits": len(res),
             "accuracy_mean": float(acc_s.mean()), "accuracy_std": float(acc_s.std()),
             "accuracy_min": float(acc_s.min()), "accuracy_max": float(acc_s.max()),
             "macro_f1_mean": float(f1_s.mean()), "macro_f1_std": float(f1_s.std()),
             "roc_auc_mean": float(auc_s.mean()), "roc_auc_std": float(auc_s.std()),
             "worst_split": worst["name"],
             "held_out_cluster_accuracy": {c: {"mean": float(np.mean(v)), "min": float(np.min(v)), "n": len(v)}
                                           for c, v in held.items()}}
    paper_split = next((r for r in res if sorted(r["test"]) == sorted(PAPER_TEST)), None)
    if paper_split is not None:
        rank = int((acc_s > paper_split["accuracy"]).sum()) + 1
        print(f"  paper's split: acc {paper_split['accuracy']:.4f}, rank {rank} of {len(res)} (1 = best)")
        entry["paper_split_accuracy"] = paper_split["accuracy"]
        entry["paper_split_rank"] = rank
    SUMMARY[protocol] = entry
    print()

## Step 20: Plots and saved summary

1. **Accuracy of every split**, sorted. The paper's split is highlighted.
2. **Accuracy of each cluster when held out:** mean, with the range from worst to best split.

In [ ]:
for protocol, res in ALL_RESULTS.items():
    acc_s = np.array([r["accuracy"] for r in res])
    order = np.argsort(acc_s)
    is_paper = np.array([sorted(r["test"]) == sorted(PAPER_TEST) for r in res])[order]

    fig, ax = plt.subplots(figsize=(8, 3.8))
    ax.plot(np.arange(len(acc_s)), acc_s[order], "o", color=COLOR_LOS, markersize=6)
    if is_paper.any():
        k = int(np.where(is_paper)[0][0])
        ax.plot(k, acc_s[order][k], "o", color=COLOR_NLOS, markersize=10)
        ax.annotate("paper's split", (k, acc_s[order][k]), xytext=(0, -62), textcoords="offset points",
                    ha="center", color=INK, arrowprops={"arrowstyle": "-", "color": INK_2})
    ax.axhline(acc_s.mean(), color=INK_2, linestyle="--", linewidth=1)
    ax.text(0, acc_s.mean(), f" mean {acc_s.mean():.3f}", va="bottom", color=INK_2)
    ax.xaxis.set_major_locator(plt.MaxNLocator(integer=True))
    ax.set_xlabel("Split (sorted by accuracy)")
    ax.set_ylabel("Test accuracy")
    ax.set_title(f"RF: accuracy of all {len(acc_s)} {protocol} splits")
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, f"splits_{protocol}_accuracy.png"), dpi=150)
    plt.show()

    held = SUMMARY[protocol]["held_out_cluster_accuracy"]
    shown = [c for c in ALL_CLUSTERS if c in held]
    means = np.array([held[c]["mean"] for c in shown])
    mins = np.array([held[c]["min"] for c in shown])
    maxs = np.array([max(r["per_cluster_accuracy"][c] for r in res if c in r["per_cluster_accuracy"]) for c in shown])
    colors = [COLOR_LOS if c.startswith("LOS") else COLOR_NLOS for c in shown]

    fig, ax = plt.subplots(figsize=(8, 4.6))
    y = np.arange(len(shown))
    ax.set_axisbelow(True)
    ax.grid(axis="y", visible=False)
    ax.barh(y, means, color=colors, height=0.6, edgecolor="#fcfcfb", linewidth=2)
    ax.errorbar(means, y, xerr=[means - mins, maxs - means], fmt="none", ecolor=INK_2, capsize=3, linewidth=1)
    for yi, m, hi in zip(y, means, maxs):
        ax.text(hi + 0.015, yi, f"{m:.3f}", va="center", color=INK, fontsize=9)
    ax.set_yticks(y, [short(c) for c in shown])
    ax.invert_yaxis()
    ax.set_xlim(0, 1.12)
    ax.set_xlabel("Accuracy when the cluster is held out (mean; bar = worst to best split)")
    ax.set_title(f"RF ({protocol}): accuracy per held-out cluster")
    ax.legend(handles=[plt.Rectangle((0, 0), 1, 1, color=COLOR_LOS), plt.Rectangle((0, 0), 1, 1, color=COLOR_NLOS)],
              labels=["LOS cluster", "NLOS cluster"], frameon=False, ncol=2,
              loc="upper center", bbox_to_anchor=(0.5, -0.16))
    fig.tight_layout()
    fig.savefig(os.path.join(RESULTS_DIR, f"splits_{protocol}_per_cluster.png"), dpi=150)
    plt.show()

with open(os.path.join(RESULTS_DIR, "splits_summary.json"), "w") as f:
    json.dump({"settings": {"n_estimators": N_ESTIMATORS, "criterion": CRITERION, "max_features": MAX_FEATURES,
                            "train_subsample": TRAIN_SUBSAMPLE, "seed": SEED},
               "protocols": SUMMARY}, f, indent=2)
print("saved:", os.path.join(RESULTS_DIR, "splits_summary.json"))

## Step 21: Comparison with the SVM on the same splits

If the SVM's Part B results exist (`../SVM/results/splits_<protocol>.jsonl`), both models are compared **split by split**. They use exactly the same 80 splits, so the comparison is paired:
- mean ± std for each model;
- the mean difference (RF − SVM) and how many splits each model wins;
- accuracy per held-out cluster for both models.

In [ ]:
for protocol, res in ALL_RESULTS.items():
    svm_path = os.path.join(SVM_RESULTS_DIR, f"splits_{protocol}.jsonl")
    if not os.path.exists(svm_path):
        print(f"{protocol}: SVM results not found ({svm_path}) -- skipped")
        continue
    svm_res = load_done(svm_path)
    common = [r for r in res if r["name"] in svm_res]
    rf_acc = np.array([r["accuracy"] for r in common])
    svm_acc = np.array([svm_res[r["name"]]["accuracy"] for r in common])
    diff = rf_acc - svm_acc

    print(f"=== {protocol}: {len(common)} common splits")
    print(f"  SVM accuracy : {svm_acc.mean():.4f} ± {svm_acc.std():.4f}   (min {svm_acc.min():.4f})")
    print(f"  RF  accuracy : {rf_acc.mean():.4f} ± {rf_acc.std():.4f}   (min {rf_acc.min():.4f})")
    print(f"  RF - SVM     : {diff.mean():+.4f} on average;  RF better on {(diff > 0).sum()}, "
          f"SVM better on {(diff < 0).sum()}, equal on {(diff == 0).sum()} splits")

    print(f"  {'held-out cluster':16s} {'SVM':>8s} {'RF':>8s}")
    for c in ALL_CLUSTERS:
        s_v = [svm_res[r["name"]]["per_cluster_accuracy"][c] for r in common if c in r["per_cluster_accuracy"]]
        r_v = [r["per_cluster_accuracy"][c] for r in common if c in r["per_cluster_accuracy"]]
        if r_v:
            print(f"  {c:16s} {np.mean(s_v):8.4f} {np.mean(r_v):8.4f}")
    print()

## Notes

- **Report Part B as the main result.** Part A is a single split, as in the paper. For the SVM, the paper's split ranked 6th of 60.
- **SNR differs by class.** LOS is at about 38 dB SNR and NLOS at about 19 dB (`../AUDIT.md`). No normalisation option removes this, so a high accuracy must be confirmed with an SNR-equalised test.
- **Fully grown trees** reach about 100% training accuracy by design; that alone is not overfitting. Compare the OOB and test accuracy in Step 12.
- **Feature importance** (Step 11) is computed on the training clusters of Part A only.
- **Interference spur at subcarriers 12–14.** An analysis run (leave-one-pair-out, 8,000 training frames per split) found that the RF depends on it much more than the SVM:

  | Model | All 52 subcarriers | Without subcarriers 12–14 |
  |---|---|---|
  | SVM | 90.9% ± 9.2% | 90.0% ± 9.8% |
  | RF | 95.3% ± 6.9% | **88.2% ± 11.8%** |

  So the RF's apparent advantage over the SVM comes from the spur. Results without these subcarriers are the ones to report.
- **Hyperparameters** follow the paper (100 trees, Gini) and scikit-learn's defaults otherwise. Tuning `max_depth`, `min_samples_leaf` or `max_features` must use cluster-wise validation, never the test clusters.